In [1]:
import geopandas as gpd
import pandas as pd
from multiprocessing import Pool, cpu_count
from functools import partial
from tqdm import tqdm
import os
from helpers.helpers import (calculate_canopy_for_all_reaches, process_canopy_chunk)
import numpy as np
import os, time

In [2]:
ta = gpd.read_file('data/chc-boundaries/territorial-authority-2021-generalised.gpkg', engine='pyogrio')
sa2 = gpd.read_file('data/chc-boundaries/sa2/statistical-area-2-2023-generalised.shp')
ta_chc = ta[ta['TA2021_V1_00_NAME_ASCII'] == 'Christchurch City']
sa2_chc = gpd.clip(sa2, ta_chc)

canopy = gpd.read_file('data/canopy.gdb')
property_full = gpd.read_file('output/property.gpkg')
access_points = gpd.read_file('output/property_accesspoints.gpkg', engine='pyogrio')

# boundary = sa2_chc[sa2_chc['SA22023__2'].str.lower().str.contains('ilam')]

property = gpd.clip(property_full, sa2_chc)
TARGET_CRS = 2193

/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


In [3]:
property_geom = property[['property_id', 'geometry']].copy()

property_geom = gpd.GeoDataFrame(
  property_geom,
  crs=property.crs,
  geometry="geometry"
)

access_points = access_points.rename(columns={'geometry': 'access_point'})

property_reaches = property_geom.merge(
    access_points[['property_id', 'access_point']],    
    on='property_id',
    how='left'
)

REACHES = [50, 100, 150, 200, 250, 300, 350, 400]

for distance in REACHES:
  path = f'output/property_reach_{distance}m.gpkg'
  reach_col = f'reach_{distance}m'
  
  if os.path.exists(path):
      reach_gdf = gpd.read_file(path, engine='pyogrio')
      reach_gdf = reach_gdf.rename(columns={'geometry': reach_col})
      
      property_reaches = property_reaches.merge(
          reach_gdf[['property_id', reach_col]],    
          on='property_id',
          how='left'
      )

In [ ]:
n_cores = cpu_count()
n_chunks = n_cores * 2

chunks = np.array_split(property_reaches, n_chunks)

In [6]:
canopy_sindex = canopy.sindex
reaches = [50, 100, 150, 200, 250, 300, 350, 400]

if __name__ == "__main__":
    with Pool(processes=n_cores) as pool:
        func = partial(
            process_canopy_chunk,
            canopy_gdf=canopy,
            canopy_sindex=canopy_sindex,
            reaches=reaches,
            buffer_dist=10
        )

        results = list(
            tqdm(
                pool.imap(func, chunks),
                total=len(chunks),
                desc="Parallel canopy calculation"
            )
        )


Parallel canopy calculation:   0%|          | 0/20 [00:00<?, ?it/s]

[PID 32505] 20/616 rows | 2.1s elapsed | 9.66 rows/s
[PID 32505] 40/616 rows | 4.6s elapsed | 8.61 rows/s
[PID 32505] 60/616 rows | 6.4s elapsed | 9.41 rows/s
[PID 32505] 80/616 rows | 7.7s elapsed | 10.36 rows/s
[PID 32505] 100/616 rows | 9.1s elapsed | 10.96 rows/s
[PID 32505] 120/616 rows | 10.0s elapsed | 11.96 rows/s
[PID 32505] 140/616 rows | 10.5s elapsed | 13.37 rows/s
[PID 32505] 160/616 rows | 11.4s elapsed | 14.01 rows/s
[PID 32505] 180/616 rows | 12.2s elapsed | 14.76 rows/s
[PID 32505] 200/616 rows | 13.3s elapsed | 15.04 rows/s
[PID 32505] 220/616 rows | 14.5s elapsed | 15.13 rows/s
[PID 32505] 240/616 rows | 16.2s elapsed | 14.85 rows/s
[PID 32505] 260/616 rows | 17.4s elapsed | 14.91 rows/s
[PID 32505] 280/616 rows | 18.7s elapsed | 14.96 rows/s
[PID 32505] 300/616 rows | 20.0s elapsed | 15.03 rows/s
[PID 32505] 320/616 rows | 20.3s elapsed | 15.80 rows/s
[PID 32505] 340/616 rows | 20.8s elapsed | 16.35 rows/s
[PID 32505] 360/616 rows | 21.6s elapsed | 16.69 rows/s
[PID

Parallel canopy calculation:   5%|▌         | 1/20 [01:24<26:49, 84.69s/it]

[PID 32506] 20/616 rows | 1.1s elapsed | 17.59 rows/s
[PID 32506] 40/616 rows | 2.6s elapsed | 15.12 rows/s
[PID 32506] 60/616 rows | 5.0s elapsed | 12.09 rows/s
[PID 32506] 80/616 rows | 6.6s elapsed | 12.16 rows/s
[PID 32506] 100/616 rows | 7.9s elapsed | 12.71 rows/s
[PID 32506] 120/616 rows | 8.8s elapsed | 13.68 rows/s
[PID 32506] 140/616 rows | 9.3s elapsed | 15.04 rows/s
[PID 32506] 160/616 rows | 9.8s elapsed | 16.40 rows/s
[PID 32506] 180/616 rows | 10.4s elapsed | 17.35 rows/s
[PID 32506] 200/616 rows | 11.1s elapsed | 18.08 rows/s
[PID 32506] 220/616 rows | 11.9s elapsed | 18.46 rows/s
[PID 32506] 240/616 rows | 13.1s elapsed | 18.27 rows/s
[PID 32506] 260/616 rows | 14.4s elapsed | 18.11 rows/s
[PID 32506] 280/616 rows | 15.6s elapsed | 17.89 rows/s
[PID 32506] 300/616 rows | 17.3s elapsed | 17.38 rows/s
[PID 32506] 320/616 rows | 18.2s elapsed | 17.55 rows/s
[PID 32506] 340/616 rows | 18.8s elapsed | 18.11 rows/s
[PID 32506] 360/616 rows | 19.4s elapsed | 18.55 rows/s
[PID

Parallel canopy calculation:  10%|█         | 2/20 [02:41<23:58, 79.91s/it]

...

Parallel canopy calculation:  90%|█████████ | 18/20 [31:23<03:31, 105.91s/it]

[PID 32513] 20/615 rows | 1.8s elapsed | 11.38 rows/s
[PID 32513] 40/615 rows | 2.8s elapsed | 14.17 rows/s
[PID 32513] 60/615 rows | 3.8s elapsed | 15.72 rows/s
[PID 32513] 80/615 rows | 6.0s elapsed | 13.24 rows/s
[PID 32513] 100/615 rows | 8.5s elapsed | 11.71 rows/s
[PID 32513] 120/615 rows | 10.9s elapsed | 10.99 rows/s
[PID 32513] 140/615 rows | 13.3s elapsed | 10.51 rows/s
[PID 32513] 160/615 rows | 15.8s elapsed | 10.10 rows/s
[PID 32513] 180/615 rows | 18.8s elapsed | 9.56 rows/s
[PID 32513] 200/615 rows | 21.8s elapsed | 9.19 rows/s
[PID 32513] 220/615 rows | 24.5s elapsed | 8.97 rows/s
[PID 32513] 240/615 rows | 28.0s elapsed | 8.58 rows/s
[PID 32513] 260/615 rows | 30.6s elapsed | 8.50 rows/s
[PID 32513] 280/615 rows | 34.7s elapsed | 8.07 rows/s
[PID 32513] 300/615 rows | 40.1s elapsed | 7.49 rows/s
[PID 32513] 320/615 rows | 44.0s elapsed | 7.27 rows/s
[PID 32513] 340/615 rows | 47.9s elapsed | 7.10 rows/s
[PID 32513] 360/615 rows | 51.8s elapsed | 6.95 rows/s
[PID 32513]

Parallel canopy calculation:  95%|█████████▌| 19/20 [33:38<01:54, 114.67s/it]

[PID 32514] 160/615 rows | 10.7s elapsed | 15.02 rows/s
[PID 32514] 180/615 rows | 11.7s elapsed | 15.43 rows/s
[PID 32514] 200/615 rows | 13.1s elapsed | 15.28 rows/s
[PID 32514] 220/615 rows | 14.8s elapsed | 14.90 rows/s
[PID 32514] 240/615 rows | 16.6s elapsed | 14.50 rows/s
[PID 32514] 260/615 rows | 17.6s elapsed | 14.77 rows/s
[PID 32514] 280/615 rows | 19.4s elapsed | 14.46 rows/s
[PID 32514] 300/615 rows | 20.7s elapsed | 14.46 rows/s
[PID 32514] 320/615 rows | 21.8s elapsed | 14.69 rows/s
[PID 32514] 340/615 rows | 22.5s elapsed | 15.12 rows/s
[PID 32514] 360/615 rows | 23.8s elapsed | 15.13 rows/s
[PID 32514] 380/615 rows | 25.2s elapsed | 15.06 rows/s
[PID 32514] 400/615 rows | 26.2s elapsed | 15.27 rows/s
[PID 32514] 420/615 rows | 26.5s elapsed | 15.85 rows/s
[PID 32514] 440/615 rows | 27.0s elapsed | 16.30 rows/s
[PID 32514] 460/615 rows | 27.4s elapsed | 16.81 rows/s
[PID 32514] 480/615 rows | 28.9s elapsed | 16.60 rows/s
[PID 32514] 500/615 rows | 31.2s elapsed | 16.04

Parallel canopy calculation: 100%|██████████| 20/20 [34:08<00:00, 102.43s/it]


In [7]:
for chunk_result in results:
    for idx, res in chunk_result:
        if res is None:
            continue
        for key, value in res.items():
            property_reaches.at[idx, key] = value


In [8]:
reaches = [50, 100, 150, 200, 250, 300, 350, 400]

canopy_cols = [f"canopy_{d}m" for d in reaches]

canopy_out = property_reaches[canopy_cols].copy()

canopy_out["property_id"] = property_reaches.index

canopy_out = canopy_out[["property_id"] + canopy_cols]

canopy_out = gpd.GeoDataFrame(
    canopy_out,
    geometry=property_reaches.geometry,
    crs=property_reaches.crs
)


In [9]:
out_path = "output/property_isodistance_canopies.gpkg"

canopy_out.to_file(
    out_path,
    layer="canopy_by_reach",
    driver="GPKG",
    engine="pyogrio"
)

In [10]:
canopy_out.shape

(12317, 10)

In [11]:
canopy_out.head()

,property_id,canopy_50m,canopy_100m,canopy_150m,canopy_200m,canopy_250m,canopy_300m,canopy_350m,canopy_400m,geometry
0,0,25.782986,192.711307,204.768643,319.550849,550.032059,682.414635,932.266395,1242.955144,POINT (1564919.129 5174157.305)
1,1,30.495473,116.039225,369.255412,501.136692,852.088438,1146.041794,1603.443922,2132.706457,POINT (1565047.92 5174178.753)
2,2,146.128014,161.708964,190.194919,204.768643,251.751149,496.983665,679.576348,856.535574,POINT (1564904.931 5174179.536)
3,3,30.495473,165.048229,353.665378,511.542446,909.228096,1135.821806,1627.178436,2312.325189,POINT (1565029.188 5174201.753)
4,4,42.975251,188.162874,194.655967,222.928693,385.847184,632.951004,767.283211,1057.061872,POINT (1564988.184 5174211.698)
